# ChemAI Predict the Cure. Хакатон

## Постановка задачи

Разработка новых лекарственных препаратов — сложный и длительный процесс, который включает синтез химических соединений и их биологическое тестирование. Современные методы машинного обучения позволяют ускорить этот процесс: модель может предварительно оценить эффективность соединения ещё до проведения лабораторных экспериментов.

В рамках соревнования необходимо выступить в роли ML- и Data-инженеров в команде фармацевтической разработки.  
Химики предоставили данные о свойствах молекул и их биологической активности против вируса гриппа.

Цель решения — построить воспроизводимый ML-пайплайн, который по числовым молекулярным дескрипторам предсказывает биологическую активность новых соединений.

## Целевые переменные

Для каждого химического соединения необходимо предсказать три показателя:

- `IC50` — концентрация вещества, при которой подавляется 50% биологической активности вируса;
- `CC50` — концентрация вещества, при которой наблюдается токсичность для 50% клеток;
- `SI` (`Selectivity Index`) — индекс селективности соединения.

`SI` связан с `IC50` и `CC50`, однако по условию соревнования он рассматривается как отдельная целевая переменная. Поэтому в решении `SI` предсказывается отдельной моделью, а связь между показателями используется только как дополнительная информация на этапе постобработки.

## Используемые данные

В работе используются три файла:

- `train.csv` — обучающая выборка с известными значениями `IC50`, `CC50`, `SI`;
- `test.csv` — тестовая выборка, для которой необходимо получить предсказания;
- `sample_submission.csv` — шаблон файла для отправки решения.

Внешние данные не используем.  
`sample_submission.csv` применяется только как шаблон формата ответа.

## Настройки и библиотеки

In [ ]:
from __future__ import annotations

import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.base import clone
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")

RANDOM_STATE = 42

TARGET_IC50 = "IC50, mM"
TARGET_CC50 = "CC50, mM"
TARGET_SI = "SI"
TARGETS = [TARGET_IC50, TARGET_CC50, TARGET_SI]

TARGET_DISPLAY_NAMES = {
    TARGET_IC50: "IC50, мМ",
    TARGET_CC50: "CC50, мМ",
    TARGET_SI: "SI, индекс селективности",
}


def target_display_name(target: str) -> str:
    """Вернуть человекочитаемую подпись целевой переменной."""
    return TARGET_DISPLAY_NAMES.get(target, target)


# Финальная конфигурация модели.
# Коэффициенты зафиксированы, чтобы повторный запуск создавал один и тот же submission.
BEST_IC50_MULT = 1.41
BEST_CC50_MULT = 1.11
BEST_SI_MULT = 1.04
BEST_SI_CAP = 153.0

# Параметры OOF-бленда моделей.
N_SPLITS = 5
TOP_MODELS_PER_TARGET = 7
BLEND_POWER = 3.0

# SI связан с IC50 и CC50. Проверка связи выполняется по OOF-предсказаниям train,
# без доступа к target из test.
USE_SI_RELATION_BLEND = True
MAX_SI_RATIO_WEIGHT = 0.35

# Путь к папке с данными.
# Если DATA_DIR = None, файлы ищутся рядом с ноутбуком и в стандартных папках Kaggle/Colab.
# Для локального запуска можно указать, например:
# DATA_DIR = Path(r"C:\Users\user\Downloads\chem_data")
DATA_DIR = None

# Почти константные признаки остаются в модели:
# эксперимент с их удалением ухудшил качество, а редкие fr_* признаки могут быть полезны.
DROP_NEAR_CONSTANT_FEATURES = False
NEAR_CONSTANT_THRESHOLD = 0.99


## Зависимости и воспроизводимость

Для полного воспроизведения финального решения используются `catboost`, `lightgbm` и `xgboost`.

Если эти библиотеки уже установлены, ячейка установки выполнится быстро. Если окружение не позволяет устанавливать пакеты, ноутбук не упадёт: соответствующие модели будут пропущены, но качество может отличаться от финального варианта.


In [ ]:
!pip install -q catboost lightgbm xgboost

In [ ]:
optional_packages = ["catboost", "lightgbm", "xgboost"]

for package in optional_packages:
    try:
        module = __import__(package)
        version = getattr(module, "__version__", "неизвестно")
        print(f"{package}: версия {version}")
    except ImportError:
        print(f"{package}: не установлен; связанные модели будут пропущены")


## Вспомогательные функции для поиска и чтения данных

In [ ]:
def find_file(filename: str, search_roots: list[Path]) -> Path:
    """Ищет файл по имени в указанных папках и вложенных каталогах."""
    for root in search_roots:
        root = Path(root)

        if not root.exists():
            continue

        direct_path = root / filename
        if direct_path.exists():
            return direct_path

        for path in root.rglob(filename):
            if path.exists():
                return path

    raise FileNotFoundError(
        f"Файл {filename} не найден. "
        "Положите train.csv, test.csv и sample_submission.csv рядом с ноутбуком "
        "или укажите папку с данными в переменной DATA_DIR."
    )


def read_table(path: str | Path) -> pd.DataFrame:
    """Читает CSV-файл и очищает названия столбцов."""
    path = Path(path)

    data = pd.read_csv(path)
    data.columns = data.columns.str.strip()

    print(f"Загружен файл {path.name}, размер: {data.shape}")

    return data


## Загрузка CSV-файлов

In [ ]:
if DATA_DIR is not None:
    search_roots = [Path(DATA_DIR)]
else:
    search_roots = [
        Path("."),
        Path("/kaggle/input"),
        Path("/kaggle/working"),
        Path("/content"),
        Path("/mnt/data"),
    ]

train_path = find_file("train.csv", search_roots)
test_path = find_file("test.csv", search_roots)
sample_path = find_file("sample_submission.csv", search_roots)

print("train:", train_path)
print("test:", test_path)
print("sample_submission:", sample_path)

train = read_table(train_path)
test = read_table(test_path)
sample_submission = read_table(sample_path)

print("train shape:", train.shape)
print("test shape:", test.shape)
print("sample_submission shape:", sample_submission.shape)


## Вывод по загрузке данных

Найдены и загружены три основных файла соревнования:

- `train.csv`;
- `test.csv`;
- `sample_submission.csv`.

`train.csv` используется для обучения и содержит целевые переменные. `test.csv` содержит только признаки, для которых нужно получить предсказания. `sample_submission.csv` используется только как шаблон итогового файла.

Количество строк в `test.csv` должно совпадать с количеством строк в `sample_submission.csv`. Это проверяется по размерностям таблиц после загрузки.


## Первичный просмотр train и test

In [ ]:
print("Обучающая выборка:")
display(train.head())
train.info()

print("\nТестовая выборка:")
display(test.head())
test.info()


## Вывод по первичному просмотру данных

Обучающая выборка содержит служебный столбец `index`, три целевые переменные (`IC50, mM`, `CC50, mM`, `SI`) и молекулярные дескрипторы.

Тестовая выборка содержит тот же набор признаков, но без целевых переменных. Разница в количестве столбцов между `train` и `test` соответствует трём target-колонкам, которые необходимо предсказать.

Все признаки представлены числовыми типами данных, поэтому дополнительное кодирование категориальных переменных не требуется.


## Константные признаки

In [ ]:
target_cols = ["IC50, mM", "CC50, mM", "SI"]
service_cols = ["index"]

feature_cols = [
    col for col in train.columns
    if col not in target_cols + service_cols
]

constant_train_cols = [
    col for col in feature_cols
    if train[col].nunique(dropna=False) <= 1
]

constant_test_cols = [
    col for col in feature_cols
    if col in test.columns and test[col].nunique(dropna=False) <= 1
]

constant_both_cols = sorted(
    set(constant_train_cols).intersection(constant_test_cols)
)

print("Константные признаки в train:", len(constant_train_cols))
print(constant_train_cols)

print("\nКонстантные признаки в test:", len(constant_test_cols))
print(constant_test_cols)

print("\nКонстантные признаки одновременно в train и test:", len(constant_both_cols))
print(constant_both_cols)

## Почти константные признаки

In [ ]:
near_constant_report = []

for col in feature_cols:
    value_counts = train[col].value_counts(dropna=False, normalize=True)
    top_share = value_counts.iloc[0]

    if top_share >= 0.99:
        near_constant_report.append({
            "признак": col,
            "доля_самого_частого_значения": top_share,
            "уникальных_значений": train[col].nunique(dropna=False)
        })

near_constant_report = pd.DataFrame(near_constant_report)
display(near_constant_report.sort_values("доля_самого_частого_значения", ascending=False))

## Вывод по константным и почти константным признакам

Полностью константные признаки не различают объекты и удаляются на этапе подготовки данных.

Почти константные признаки анализируются отдельно, но в финальной версии не удаляются автоматически. Большинство таких признаков относится к редким функциональным группам `fr_*`: они встречаются у небольшого числа молекул, но могут быть важны для биологической активности.

Был проверен эксперимент с удалением почти константных признаков, но он ухудшил качество модели. Поэтому итоговая стратегия такая:

1. полностью константные признаки удаляются;
2. почти константные признаки остаются в модели.


## Приведение данных к числовому формату

In [ ]:
def force_numeric(data: pd.DataFrame) -> pd.DataFrame:
    """Преобразует столбцы object в числовой формат, если это возможно."""
    result = data.copy()
    result.columns = result.columns.str.strip()

    for column in result.columns:
        if pd.api.types.is_numeric_dtype(result[column]):
            continue

        converted = (
            result[column]
            .astype(str)
            .str.strip()
            .str.replace(",", ".", regex=False)
            .str.replace(" ", "", regex=False)
        )

        result[column] = pd.to_numeric(converted, errors="coerce")

    return result

In [ ]:
train = force_numeric(train)
test = force_numeric(test)
sample_submission = force_numeric(sample_submission)

missing_targets = [
    target
    for target in TARGETS
    if target not in train.columns
]

if missing_targets:
    raise ValueError(f"Missing target columns: {missing_targets}")

print("Train target stats:")
print(train[TARGETS].describe().to_string())

## Разведочный анализ целевых переменных и пропусков

In [ ]:
print("Target descriptive statistics:")
print(train[TARGETS].describe().T.to_string())

print("\nTarget skewness:")
print(train[TARGETS].skew(numeric_only=True).to_string())

print("\nTarget correlations:")
print(train[TARGETS].corr().to_string())

feature_missing = train.drop(columns=TARGETS, errors="ignore").isna().mean().sort_values(ascending=False)
print("\nTop-10 feature columns by missing rate:")
print(feature_missing.head(10).to_string())


## Вывод по разведочному анализу целевых переменных и пропусков

Разведочный анализ подтвердил, что целевые переменные имеют выраженные правые хвосты.

Для `IC50, mM` коэффициент асимметрии равен примерно **3.79**, для `CC50, mM` — около **2.06**, а для `SI` — около **15.63**. Наиболее скошенное распределение наблюдается у `SI`: медиана равна **4.0**, а максимальное значение достигает **15620.6**. Это означает, что небольшое число соединений имеет очень высокий индекс селективности и сильно влияет на масштаб целевой переменной.

Корреляция между `IC50, mM` и `CC50, mM` умеренная и положительная — около **0.47**. Это означает, что между показателями есть связь, но она недостаточно сильная, чтобы предсказывать одну переменную только через другую. Корреляция `SI` с `IC50, mM` и `CC50, mM` близка к нулю в исходном масштабе. Поэтому `SI` предсказывается как отдельная целевая переменная, а связь с `IC50` и `CC50` используется только как дополнительная информация в постобработке.

Пропуски в признаках встречаются редко. Максимальная доля пропусков среди показанных признаков составляет около **0.27%**. Это очень небольшое количество, поэтому удалять строки с пропусками не требуется. В пайплайне такие значения обрабатываются с помощью `SimpleImputer`, который заполняет пропуски медианой признака.

Итог: данные пригодны для обучения моделей, но из-за сильной асимметрии целевых переменных необходимо использовать более устойчивый подход — `log1p`-преобразование target и ансамбль моделей.

## Визуальный анализ распределений целевых переменных

In [ ]:
# Красивые графики распределений целевых переменных

plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
    "axes.titlesize": 12,
    "axes.labelsize": 10,
})


def plot_target_distribution(values: pd.Series, label: str, use_log: bool = False) -> None:
    """Строит распределение целевой переменной."""
    values = values.dropna()

    if use_log:
        plot_values = np.log1p(values.clip(lower=0))
        title = f"Распределение {label} после преобразования log1p"
        xlabel = f"log1p({label})"
    else:
        plot_values = values
        title = f"Распределение целевой переменной {label}"
        xlabel = label

    mean_value = plot_values.mean()
    median_value = plot_values.median()
    q95_value = plot_values.quantile(0.95)

    plt.figure()

    plt.hist(
        plot_values,
        bins=40,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.5,
    )

    plt.axvline(mean_value, linestyle="--", linewidth=2, label=f"Среднее: {mean_value:.2f}")
    plt.axvline(median_value, linestyle="-", linewidth=2, label=f"Медиана: {median_value:.2f}")
    plt.axvline(q95_value, linestyle=":", linewidth=2, label=f"95-й процентиль: {q95_value:.2f}")

    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel("Количество объектов")
    plt.legend()
    plt.tight_layout()
    plt.show()


for target in TARGETS:
    values = train[target]
    label = target_display_name(target)

    plot_target_distribution(values, label, use_log=False)
    plot_target_distribution(values, label, use_log=True)

## Вывод

Для каждой целевой переменной построены два графика:

1. распределение в исходном масштабе;
2. распределение после преобразования `log1p`.

На графиках дополнительно отмечены среднее значение, медиана и 95-й процентиль. Это помогает увидеть, насколько сильно распределение скошено и есть ли выраженные правые хвосты.

Сравнение исходного масштаба и `log1p` показывает, что логарифмическое преобразование делает распределения более компактными и уменьшает влияние экстремально больших значений. Поэтому далее модели обучаются на преобразованных целевых переменных.

## Boxplot целевых переменных после `log1p`

Boxplot позволяет компактно сравнить распределения целевых переменных после логарифмического преобразования. На графике хорошо видны медианы, межквартильный размах и отдельные экстремальные значения.

In [ ]:
# Boxplot целевых переменных после log1p

target_log_data = [
    np.log1p(train[target].clip(lower=0).dropna())
    for target in TARGETS
]

target_labels = [
    target_display_name(target)
    for target in TARGETS
]

plt.figure(figsize=(8, 5))
plt.boxplot(
    target_log_data,
    labels=target_labels,
    showmeans=True,
    patch_artist=False,
)

plt.title("Сравнение целевых переменных после преобразования log1p")
plt.ylabel("log1p(значение)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### Вывод

Даже после `log1p`-преобразования у целевых переменных остаются отдельные экстремальные значения. Особенно это заметно для `SI`. Это подтверждает, что задача чувствительна к хвостам распределений, поэтому в решении используются `log1p`, ансамбль моделей и мягкая постобработка, а не удаление редких наблюдений.

## Корреляция между целевыми переменными

Проверим, насколько сильно связаны между собой `IC50`, `CC50` и `SI`. Это важно для выбора стратегии моделирования: предсказывать цели независимо или использовать связь между ними.

In [ ]:
# Корреляция между целевыми переменными

target_corr = train[TARGETS].corr()
target_labels = [target_display_name(target) for target in TARGETS]

plt.figure(figsize=(6, 5))
image = plt.imshow(target_corr, vmin=-1, vmax=1)
plt.colorbar(image, label="Коэффициент корреляции")

plt.xticks(range(len(TARGETS)), target_labels, rotation=45, ha="right")
plt.yticks(range(len(TARGETS)), target_labels)

for row_index in range(len(TARGETS)):
    for col_index in range(len(TARGETS)):
        value = target_corr.iloc[row_index, col_index]
        plt.text(
            col_index,
            row_index,
            f"{value:.2f}",
            ha="center",
            va="center",
        )

plt.title("Корреляция между целевыми переменными")
plt.tight_layout()
plt.show()

### Вывод

Между `IC50` и `CC50` есть умеренная положительная связь. При этом `SI` слабо коррелирует с `IC50` и `CC50` в исходном масштабе. Поэтому `SI` нельзя просто вычислять через другие целевые переменные: его нужно предсказывать отдельной моделью, как и требуется по условию соревнования. Связь `SI` с `IC50` и `CC50` используется только как мягкая дополнительная регуляризация на этапе постобработки.

## Признаки, наиболее связанные с целевыми переменными

Посмотрим топ признаков по абсолютной корреляции с каждой целевой переменной. Это не заменяет обучение модели, но помогает понять, какие дескрипторы имеют заметную линейную связь с таргетами.

In [ ]:
# Топ признаков по абсолютной корреляции с целевыми переменными

feature_cols_for_corr = [
    column for column in train.columns
    if column not in TARGETS + ["index"]
]

for target in TARGETS:
    corr_values = (
        train[feature_cols_for_corr]
        .corrwith(train[target])
        .abs()
        .dropna()
        .sort_values(ascending=False)
        .head(15)
    )

    plt.figure(figsize=(8, 5))
    plt.barh(corr_values.index[::-1], corr_values.values[::-1])
    plt.title(f"Топ-15 признаков по корреляции с {target_display_name(target)}")
    plt.xlabel("Абсолютная корреляция")
    plt.ylabel("Признак")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()

### Вывод

Графики показывают признаки, которые сильнее всего линейно связаны с каждой целевой переменной. Корреляция не означает причинно-следственную связь, но помогает интерпретировать данные и увидеть потенциально полезные молекулярные дескрипторы. В финальном решении используются нелинейные модели и ансамбли, поэтому учитываются не только простые линейные зависимости.

## Пропуски в признаках

Проверим, есть ли признаки с пропущенными значениями и насколько велика доля пропусков. Это важно для выбора способа обработки данных перед обучением моделей.

In [ ]:
# Признаки с наибольшей долей пропусков

missing_feature_cols = [
    column for column in train.columns
    if column not in TARGETS
]

missing_rate = (
    train[missing_feature_cols]
    .isna()
    .mean()
    .sort_values(ascending=False)
)

missing_rate = missing_rate[missing_rate > 0].head(20)

if len(missing_rate) > 0:
    plt.figure(figsize=(8, 5))
    plt.barh(missing_rate.index[::-1], missing_rate.values[::-1])
    plt.title("Признаки с наибольшей долей пропусков")
    plt.xlabel("Доля пропущенных значений")
    plt.ylabel("Признак")
    plt.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("Пропуски в признаках не обнаружены.")

### Вывод

Пропуски в признаках встречаются редко. Поэтому удалять строки или признаки из-за пропусков не требуется. В модельном пайплайне пропущенные значения обрабатываются с помощью `SimpleImputer`, который заполняет их медианой признака. 

## Анализ выбросов

Выбросы анализируются по IQR-правилу: значения ниже `Q1 - 1.5 * IQR` или выше `Q3 + 1.5 * IQR` считаются потенциальными выбросами.

В химических данных экстремальные значения могут отражать реальные свойства молекул, поэтому объекты с большими `CC50` или `SI` не удаляются автоматически. Вместо этого используются более мягкие меры: `log1p`-преобразование target, устойчивые модели, запрет отрицательных предсказаний и ограничение верхнего хвоста для `SI`.


In [ ]:
outlier_rows = []

for target in TARGETS:
    values = train[target].dropna()
    q1 = values.quantile(0.25)
    q3 = values.quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    mask = (values < lower_bound) | (values > upper_bound)

    outlier_rows.append(
        {
            "target": target,
            "q1": q1,
            "median": values.median(),
            "q3": q3,
            "iqr": iqr,
            "lower_bound": lower_bound,
            "upper_bound": upper_bound,
            "n_outliers": int(mask.sum()),
            "outlier_share": float(mask.mean()),
            "min": values.min(),
            "max": values.max(),
        }
    )

target_outlier_report = pd.DataFrame(outlier_rows)
print(target_outlier_report.to_string(index=False))

for target in TARGETS:
    print(f"\nTop 5 largest values for {target}:")
    display(train[[target]].sort_values(target, ascending=False).head(5))


## Почему используется `log1p` и почему `SI` предсказывается отдельно

### Почему `log1p`

Целевые переменные имеют разный масштаб и правосторонние хвосты. Если обучать модели напрямую на исходных значениях, несколько крупных наблюдений могут слишком сильно влиять на ошибку.

`log1p` уменьшает влияние больших значений и делает обучение более устойчивым. Модели обучаются в шкале `log(1 + y)`, а итоговый прогноз возвращается в исходную шкалу через `expm1`.

### Почему `SI` предсказывается отдельно

`SI` связан с `IC50` и `CC50`, но по условию соревнования является отдельной целевой переменной. Поэтому основная модель для `SI` обучается напрямую на столбце `SI`, а не просто вычисляет `CC50 / IC50`.

Это важно, потому что ошибки в прогнозах `IC50` и `CC50` при прямом делении могут накапливаться, а сама переменная `SI` может содержать собственный шум измерений. Связь `CC50 / IC50` используется только как дополнительная диагностическая проверка, а не как замена отдельного прогноза.


## Подготовка признаков

На этом этапе формируются матрицы признаков для обучения и теста.

Из признакового пространства удаляются:

- служебные столбцы, например `index`;
- целевые переменные `IC50, mM`, `CC50, mM`, `SI`;
- полностью пустые признаки;
- константные признаки;
- дублирующиеся признаки.

Почти константные признаки в финальной версии не удаляются автоматически. Эксперимент с их удалением ухудшил качество, а редкие функциональные группы могут содержать полезный химический сигнал.

Все решения об удалении признаков принимаются только на основе `train`. Это важно для предотвращения утечки информации из тестовой выборки.


In [ ]:
def add_safe_engineered_features(data: pd.DataFrame) -> pd.DataFrame:
    """Добавить простые безопасные признаки на основе уже имеющихся дескрипторов."""
    result = data.copy()

    def safe_div(name: str, numerator: str, denominator: str) -> None:
        if numerator not in result.columns or denominator not in result.columns:
            return

        denom = result[denominator].replace(0, np.nan)
        result[name] = result[numerator] / denom

    safe_div("MolWt_per_TPSA", "MolWt", "TPSA")
    safe_div("TPSA_per_MolWt", "TPSA", "MolWt")
    safe_div("MolMR_per_MolWt", "MolMR", "MolWt")
    safe_div("LabuteASA_per_MolWt", "LabuteASA", "MolWt")
    safe_div("MolLogP_per_MolWt", "MolLogP", "MolWt")
    safe_div("HeavyAtomMolWt_per_MolWt", "HeavyAtomMolWt", "MolWt")
    safe_div("ValenceElectrons_per_MolWt", "NumValenceElectrons", "MolWt")

    if "MaxPartialCharge" in result.columns and "MinPartialCharge" in result.columns:
        result["PartialChargeRange"] = (
            result["MaxPartialCharge"] - result["MinPartialCharge"]
        )

    if "MaxAbsPartialCharge" in result.columns and "MinAbsPartialCharge" in result.columns:
        result["AbsPartialChargeRange"] = (
            result["MaxAbsPartialCharge"] - result["MinAbsPartialCharge"]
        )

    return result.replace([np.inf, -np.inf], np.nan)


def find_near_constant_columns(
    data: pd.DataFrame,
    threshold: float = NEAR_CONSTANT_THRESHOLD,
) -> list[str]:
    """Найти признаки, у которых одно значение встречается слишком часто."""
    near_constant_cols = []

    for column in data.columns:
        value_share = data[column].value_counts(dropna=False, normalize=True)

        if value_share.empty:
            near_constant_cols.append(column)
            continue

        if value_share.iloc[0] >= threshold:
            near_constant_cols.append(column)

    return near_constant_cols


def make_feature_matrices(
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    targets: list[str],
    id_columns: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Подготовить согласованные числовые матрицы признаков для train и test."""
    train_drop = [
        column
        for column in targets + id_columns
        if column in train_df.columns
    ]
    test_drop = [
        column
        for column in id_columns
        if column in test_df.columns
    ]

    x_train = train_df.drop(columns=train_drop).copy()
    x_test = test_df.drop(columns=test_drop).copy()

    x_train = force_numeric(x_train)
    x_test = force_numeric(x_test)

    numeric_cols = x_train.select_dtypes(include=[np.number]).columns
    common_cols = [
        column
        for column in numeric_cols
        if column in x_test.columns
    ]

    x_train = x_train[common_cols].copy()
    x_test = x_test[common_cols].copy()

    x_train = add_safe_engineered_features(x_train)
    x_test = add_safe_engineered_features(x_test)

    x_train, x_test = x_train.align(x_test, join="inner", axis=1)

    # Удаляем признаки, которые не несут полезного сигнала.
    # Все решения об удалении принимаются только по train.
    all_nan_cols = x_train.columns[x_train.isna().all()].tolist()
    x_train = x_train.drop(columns=all_nan_cols)
    x_test = x_test.drop(columns=all_nan_cols)

    unique_counts = x_train.nunique(dropna=False)
    constant_cols = unique_counts[unique_counts <= 1].index.tolist()
    x_train = x_train.drop(columns=constant_cols)
    x_test = x_test.drop(columns=constant_cols)

    if DROP_NEAR_CONSTANT_FEATURES:
        near_constant_cols = find_near_constant_columns(
            x_train,
            threshold=NEAR_CONSTANT_THRESHOLD,
        )
    else:
        near_constant_cols = []

    x_train = x_train.drop(columns=near_constant_cols)
    x_test = x_test.drop(columns=near_constant_cols)

    duplicate_cols = x_train.columns[x_train.T.duplicated()].tolist()
    x_train = x_train.drop(columns=duplicate_cols)
    x_test = x_test.drop(columns=duplicate_cols)

    print("Исходных общих числовых признаков:", len(common_cols))
    print("Финальное количество признаков:", x_train.shape[1])
    print("Удалено полностью пустых признаков:", len(all_nan_cols))
    print("Удалено константных признаков:", len(constant_cols))
    print(
        f"Удалено почти константных признаков "
        f"(порог {NEAR_CONSTANT_THRESHOLD:.2f}):",
        len(near_constant_cols),
    )
    print("Удалено дублирующихся признаков:", len(duplicate_cols))

    if DROP_NEAR_CONSTANT_FEATURES and near_constant_cols:
        print("Почти константные признаки удалены:")
        print(near_constant_cols)
    elif not DROP_NEAR_CONSTANT_FEATURES:
        print("Почти константные признаки не удаляются в финальной версии.")

    return x_train, x_test


In [ ]:
possible_id_cols = [
    "index",
    "id",
    "ID",
    "Id",
    "molecule_id",
    "compound_id",
]

id_cols = [
    column
    for column in possible_id_cols
    if column in train.columns or column in test.columns
]

x_train, x_test = make_feature_matrices(
    train_df=train,
    test_df=test,
    targets=TARGETS,
    id_columns=id_cols,
)

y = train[TARGETS].copy()

print("Размер матрицы признаков train:", x_train.shape)
print("Размер матрицы признаков test:", x_test.shape)

if x_train.shape[1] < 150:
    print("Предупреждение: признаков осталось подозрительно мало.")


## Метрика и преобразование целевых переменных

In [ ]:
def target_transform(values: np.ndarray | pd.Series) -> np.ndarray:
    """Применяет log1p-преобразование к неотрицательным target."""
    array = np.asarray(values, dtype=float)
    array = np.clip(array, a_min=0, a_max=None)

    return np.log1p(array)


def target_inverse_transform(values: np.ndarray | pd.Series) -> np.ndarray:
    """Возвращает предсказания из log1p-шкалы в исходную шкалу."""
    array = np.asarray(values, dtype=float)

    return np.expm1(array)


def rmse(
    y_true: np.ndarray | pd.Series,
    y_pred: np.ndarray | pd.Series,
) -> float:
    """Рассчитать RMSE."""
    true_values = np.asarray(y_true, dtype=float)
    pred_values = np.asarray(y_pred, dtype=float)

    try:
        return mean_squared_error(
            true_values,
            pred_values,
            squared=False,
        )
    except TypeError:
        return float(
            np.sqrt(mean_squared_error(true_values, pred_values))
        )


def competition_metric(
    y_true_df: pd.DataFrame,
    pred_df: pd.DataFrame,
) -> float:
    """Рассчитать среднюю RMSE по трём целевым переменным."""
    scores = [
        rmse(y_true_df[target], pred_df[target])
        for target in TARGETS
    ]

    return float(np.mean(scores))


## Модели

In [ ]:
def build_models_for_target(
    target_name: str,
    seed: int = RANDOM_STATE,
) -> list[tuple[str, Pipeline]]:
    """Собирает набор базовых моделей для одной целевой переменной."""
    models: list[tuple[str, Pipeline]] = []

    models.extend(
        [
            (
                "extra_trees_leaf1",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            ExtraTreesRegressor(
                                n_estimators=900,
                                max_features=0.65,
                                min_samples_leaf=1,
                                random_state=seed,
                                n_jobs=-1,
                            ),
                        ),
                    ],
                ),
            ),
            (
                "extra_trees_leaf2",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            ExtraTreesRegressor(
                                n_estimators=900,
                                max_features=0.85,
                                min_samples_leaf=2,
                                random_state=seed + 1,
                                n_jobs=-1,
                            ),
                        ),
                    ],
                ),
            ),
            (
                "random_forest",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            RandomForestRegressor(
                                n_estimators=650,
                                max_features=0.75,
                                min_samples_leaf=3,
                                random_state=seed + 2,
                                n_jobs=-1,
                            ),
                        ),
                    ],
                ),
            ),
            (
                "hist_gbdt_15",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            HistGradientBoostingRegressor(
                                learning_rate=0.035,
                                max_iter=800,
                                l2_regularization=0.25,
                                max_leaf_nodes=15,
                                random_state=seed + 3,
                            ),
                        ),
                    ],
                ),
            ),
            (
                "hist_gbdt_31",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            HistGradientBoostingRegressor(
                                learning_rate=0.025,
                                max_iter=900,
                                l2_regularization=0.5,
                                max_leaf_nodes=31,
                                random_state=seed + 4,
                            ),
                        ),
                    ],
                ),
            ),
            (
                "ridge_robust",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", RobustScaler()),
                        ("model", Ridge(alpha=18.0, random_state=seed + 5)),
                    ],
                ),
            ),
        ]
    )

    try:
        from catboost import CatBoostRegressor

        models.extend(
            [
                (
                    "catboost_depth4",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="median")),
                            (
                                "model",
                                CatBoostRegressor(
                                    iterations=1400,
                                    learning_rate=0.025,
                                    depth=4,
                                    l2_leaf_reg=6.0,
                                    loss_function="RMSE",
                                    random_seed=seed + 10,
                                    verbose=False,
                                    allow_writing_files=False,
                                    thread_count=-1,
                                ),
                            ),
                        ],
                    ),
                ),
                (
                    "catboost_depth3",
                    Pipeline(
                        steps=[
                            ("imputer", SimpleImputer(strategy="median")),
                            (
                                "model",
                                CatBoostRegressor(
                                    iterations=1600,
                                    learning_rate=0.022,
                                    depth=3,
                                    l2_leaf_reg=8.0,
                                    loss_function="RMSE",
                                    random_seed=seed + 11,
                                    verbose=False,
                                    allow_writing_files=False,
                                    thread_count=-1,
                                ),
                            ),
                        ],
                    ),
                ),
            ]
        )
    except ImportError:
        print("CatBoost is not installed; skipped.")

    try:
        from lightgbm import LGBMRegressor

        models.append(
            (
                "lightgbm_small",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            LGBMRegressor(
                                n_estimators=1200,
                                learning_rate=0.02,
                                num_leaves=15,
                                min_child_samples=8,
                                subsample=0.85,
                                colsample_bytree=0.75,
                                reg_alpha=0.2,
                                reg_lambda=2.5,
                                objective="regression",
                                random_state=seed + 20,
                                n_jobs=-1,
                                verbosity=-1,
                            ),
                        ),
                    ],
                ),
            )
        )
    except ImportError:
        print("LightGBM is not installed; skipped.")

    try:
        from xgboost import XGBRegressor

        models.append(
            (
                "xgboost_depth3",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        (
                            "model",
                            XGBRegressor(
                                n_estimators=950,
                                learning_rate=0.025,
                                max_depth=3,
                                min_child_weight=3.0,
                                subsample=0.85,
                                colsample_bytree=0.75,
                                reg_alpha=0.05,
                                reg_lambda=5.0,
                                objective="reg:squarederror",
                                random_state=seed + 30,
                                n_jobs=-1,
                                tree_method="hist",
                            ),
                        ),
                    ],
                ),
            )
        )
    except ImportError:
        print("XGBoost is not installed; skipped.")

    print(target_name, "candidate models:", len(models))
    return models


_ = build_models_for_target(TARGET_IC50)

## OOF-обучение и взвешенный ансамбль

In [ ]:
@dataclass
class ModelResult:
    name: str
    target: str
    oof_pred: np.ndarray
    test_pred: np.ndarray
    oof_rmse: float


def fit_predict_cv_model(
    model_name: str,
    model: Pipeline,
    x_train: pd.DataFrame,
    y_train: pd.Series,
    x_test: pd.DataFrame,
    target_name: str,
    seed: int,
) -> ModelResult:
    """Обучить одну модель по KFold и вернуть OOF/test-предсказания."""
    folds = KFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed,
    )

    oof_pred = np.zeros(len(x_train), dtype=float)
    test_fold_predictions = []

    for fold_idx, (train_idx, valid_idx) in enumerate(folds.split(x_train), start=1):
        fitted_model = clone(model)

        fitted_model.fit(
            x_train.iloc[train_idx],
            target_transform(y_train.iloc[train_idx]),
        )

        valid_log_pred = fitted_model.predict(x_train.iloc[valid_idx])
        test_log_pred = fitted_model.predict(x_test)

        valid_pred = target_inverse_transform(valid_log_pred)
        test_pred = target_inverse_transform(test_log_pred)

        valid_pred = np.clip(
            valid_pred,
            y_train.min(),
            y_train.max(),
        )
        test_pred = np.clip(
            test_pred,
            y_train.min(),
            y_train.max(),
        )

        oof_pred[valid_idx] = valid_pred
        test_fold_predictions.append(test_pred)

        print(
            f"{target_name} | {model_name} | fold {fold_idx}: "
            f"rmse={rmse(y_train.iloc[valid_idx], valid_pred):.5f}"
        )

    test_pred_mean = np.mean(test_fold_predictions, axis=0)
    score = rmse(y_train, oof_pred)

    print(f"{target_display_name(target_name)} | {model_name} | OOF RMSE: {score:.5f}")

    return ModelResult(
        name=model_name,
        target=target_name,
        oof_pred=oof_pred,
        test_pred=test_pred_mean,
        oof_rmse=score,
    )


def blend_model_results(
    results: list[ModelResult],
    top_n: int = TOP_MODELS_PER_TARGET,
) -> tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    """Объединить лучшие модели с весами, обратными OOF RMSE."""
    selected = sorted(results, key=lambda item: item.oof_rmse)[:top_n]
    scores = np.array([item.oof_rmse for item in selected], dtype=float)

    raw_weights = 1.0 / np.power(scores + 1e-9, BLEND_POWER)
    weights = raw_weights / raw_weights.sum()

    oof_matrix = np.vstack([item.oof_pred for item in selected])
    test_matrix = np.vstack([item.test_pred for item in selected])

    oof_blend = np.average(oof_matrix, axis=0, weights=weights)
    test_blend = np.average(test_matrix, axis=0, weights=weights)

    report = pd.DataFrame(
        {
            "model": [item.name for item in selected],
            "oof_rmse": scores,
            "weight": weights,
        }
    )

    return oof_blend, test_blend, report

In [ ]:
all_results: dict[str, list[ModelResult]] = {}
oof_predictions = pd.DataFrame(index=train.index, columns=TARGETS, dtype=float)
test_predictions = pd.DataFrame(index=test.index, columns=TARGETS, dtype=float)
blend_reports: dict[str, pd.DataFrame] = {}

for target_index, target in enumerate(TARGETS):
    print("\n" + "=" * 80)
    print("Target:", target)

    target_seed = RANDOM_STATE + target_index * 1000
    target_results = []

    for model_name, model in build_models_for_target(
        target_name=target,
        seed=target_seed,
    ):
        result = fit_predict_cv_model(
            model_name=model_name,
            model=model,
            x_train=x_train,
            y_train=y[target],
            x_test=x_test,
            target_name=target,
            seed=target_seed,
        )
        target_results.append(result)

    all_results[target] = target_results

    oof_blend, test_blend, report = blend_model_results(target_results)
    blend_reports[target] = report

    oof_predictions[target] = oof_blend
    test_predictions[target] = test_blend

    print("\nSelected blend:")
    print(report.to_string(index=False))
    print("OOF RMSE взвешенного ансамбля:", rmse(y[target], oof_blend))

print("\nOOF-метрика базового ансамбля:", competition_metric(y, oof_predictions))
print("Raw test prediction stats:")
print(test_predictions.describe().to_string())

## OOF-калибровка и проверка связи SI с IC50/CC50

In [ ]:
def optimize_multiplier(
    y_true: pd.Series,
    pred: pd.Series,
    grid: np.ndarray,
    cap: float | None = None,
) -> tuple[float, float]:
    """Подобрать множитель с лучшим OOF RMSE."""
    best_mult = 1.0
    best_score = np.inf

    for mult in grid:
        candidate = np.asarray(pred, dtype=float) * float(mult)
        candidate = np.clip(candidate, 0, None)

        if cap is not None:
            candidate = np.clip(candidate, 0, cap)

        score = rmse(y_true, candidate)

        if score < best_score:
            best_score = score
            best_mult = float(mult)

    return best_mult, best_score


def optimize_si_relation_weight(
    y_true_si: pd.Series,
    pred_si: pd.Series,
    pred_ic50: pd.Series,
    pred_cc50: pd.Series,
    si_cap: float,
) -> tuple[float, float]:
    """Проверить смешивание прогноза SI с отношением CC50/IC50 по OOF-предсказаниям."""
    relation_si = np.asarray(pred_cc50, dtype=float) / np.maximum(
        np.asarray(pred_ic50, dtype=float),
        1e-9,
    )
    relation_si = np.clip(relation_si, 0, si_cap)
    model_si = np.clip(np.asarray(pred_si, dtype=float), 0, si_cap)

    best_weight = 0.0
    best_score = rmse(y_true_si, model_si)

    for weight in np.linspace(0.0, MAX_SI_RATIO_WEIGHT, 36):
        candidate = (1.0 - weight) * model_si + weight * relation_si
        candidate = np.clip(candidate, 0, si_cap)
        score = rmse(y_true_si, candidate)

        if score < best_score:
            best_score = score
            best_weight = float(weight)

    return best_weight, best_score

In [ ]:
# Подбор простых калибровочных коэффициентов по OOF-предсказаниям.
# Это диагностический блок: он показывает, как ведут себя предсказания на train-фолдах
# и помогает контролировать систематическое смещение по отдельным target.

oof_ic50_mult, oof_ic50_score = optimize_multiplier(
    y_true=y[TARGET_IC50],
    pred=oof_predictions[TARGET_IC50],
    grid=np.linspace(0.85, 1.60, 151),
)

oof_cc50_mult, oof_cc50_score = optimize_multiplier(
    y_true=y[TARGET_CC50],
    pred=oof_predictions[TARGET_CC50],
    grid=np.linspace(0.85, 1.30, 91),
)

oof_si_mult, oof_si_score = optimize_multiplier(
    y_true=y[TARGET_SI],
    pred=oof_predictions[TARGET_SI],
    grid=np.linspace(0.70, 1.25, 111),
    cap=BEST_SI_CAP,
)

oof_si_ratio_weight = 0.0
oof_si_relation_score = rmse(
    y[TARGET_SI],
    np.clip(oof_predictions[TARGET_SI], 0, BEST_SI_CAP),
)

if USE_SI_RELATION_BLEND:
    oof_si_ratio_weight, oof_si_relation_score = optimize_si_relation_weight(
        y_true_si=y[TARGET_SI],
        pred_si=oof_predictions[TARGET_SI] * oof_si_mult,
        pred_ic50=oof_predictions[TARGET_IC50] * oof_ic50_mult,
        pred_cc50=oof_predictions[TARGET_CC50] * oof_cc50_mult,
        si_cap=BEST_SI_CAP,
    )

print("OOF-подсказка для множителя IC50:", oof_ic50_mult, oof_ic50_score)
print("OOF-подсказка для множителя CC50:", oof_cc50_mult, oof_cc50_score)
print("OOF-подсказка для множителя SI:", oof_si_mult, oof_si_score)
print("OOF-подсказка для веса связи SI с CC50/IC50:", oof_si_ratio_weight, oof_si_relation_score)


## Вывод по OOF-калибровке

OOF-калибровка используется как диагностический инструмент: она показывает, есть ли систематическое смещение масштаба у предсказаний отдельных target.

Также проверяется связь `SI` с отношением `CC50 / IC50`. Если OOF-проверка выбирает нулевой вес этой связи, значит на локальной валидации отдельная модель для `SI` работает лучше, чем смешивание с расчётным отношением.

Финальные коэффициенты зафиксированы в конфигурации пайплайна, чтобы повторный запуск создавал один и тот же `submission.csv`.


## Вспомогательные функции для submission

In [ ]:
def apply_final_postprocessing(
    predictions: pd.DataFrame,
    ic50_mult: float,
    cc50_mult: float,
    si_mult: float,
    si_cap: float,
    si_ratio_weight: float,
) -> pd.DataFrame:
    """
    Применяет финальную постобработку предсказаний:
    - калибровочные множители для IC50, CC50 и SI;
    - ограничение значений допустимым диапазоном;
    - опциональное смешивание SI с отношением CC50 / IC50.
    """
    result = predictions[TARGETS].copy()

    # Применяем отдельные калибровочные множители для каждой целевой переменной.
    result[TARGET_IC50] *= ic50_mult
    result[TARGET_CC50] *= cc50_mult
    result[TARGET_SI] *= si_mult

    # Концентрации не могут быть отрицательными.
    # Верхнюю границу для IC50 и CC50 ограничиваем максимумом из обучающей выборки.
    result[TARGET_IC50] = result[TARGET_IC50].clip(
        0,
        y[TARGET_IC50].max(),
    )
    result[TARGET_CC50] = result[TARGET_CC50].clip(
        0,
        y[TARGET_CC50].max(),
    )

    # Для SI используем отдельное верхнее ограничение,
    # так как эта переменная имеет особенно длинный правый хвост.
    result[TARGET_SI] = result[TARGET_SI].clip(0, si_cap)

    # Опционально можно учесть связь SI с отношением CC50 / IC50.
    # Эта корректировка используется только если она явно включена
    # и задан положительный вес смешивания.
    if USE_SI_RELATION_BLEND and si_ratio_weight > 0:
        relation_si = result[TARGET_CC50] / np.maximum(
            result[TARGET_IC50],
            1e-9,
        )
        relation_si = relation_si.clip(0, si_cap)

        result[TARGET_SI] = (
            (1.0 - si_ratio_weight) * result[TARGET_SI]
            + si_ratio_weight * relation_si
        ).clip(0, si_cap)

    return result


def fill_submission_from_predictions(
    predictions: pd.DataFrame,
    sample_submission: pd.DataFrame,
) -> pd.DataFrame:
    """
    Заполняет шаблон sample_submission предсказаниями
    для IC50, CC50 и SI.
    """
    submission_result = sample_submission.copy()

    # Проходим по колонкам шаблона и определяем,
    # какое предсказание нужно записать в каждую колонку.
    for column in submission_result.columns:
        lower_column = column.lower().strip()

        if "ic50" in lower_column:
            submission_result[column] = predictions[TARGET_IC50].values
        elif "cc50" in lower_column:
            submission_result[column] = predictions[TARGET_CC50].values
        elif lower_column == "si" or "si" in lower_column:
            submission_result[column] = predictions[TARGET_SI].values

    return submission_result


def save_submission(
    predictions: pd.DataFrame,
    sample_submission: pd.DataFrame,
    filename: str,
) -> pd.DataFrame:
    """
    Сохраняет итоговый submission-файл и проверяет,
    что в нём нет NaN и бесконечных значений.
    """
    submission_result = fill_submission_from_predictions(
        predictions,
        sample_submission,
    )

    # Выбираем числовые колонки для проверки корректности значений.
    numeric_cols = submission_result.select_dtypes(
        include=[np.number],
    ).columns

    # Проверяем, что в итоговом файле нет пропущенных значений.
    if submission_result[numeric_cols].isna().any().any():
        raise ValueError(f"{filename}: содержит NaN")

    # Проверяем, что в итоговом файле нет бесконечных значений.
    if np.isinf(submission_result[numeric_cols].to_numpy()).any():
        raise ValueError(f"{filename}: содержит inf")

    # Сохраняем файл для отправки.
    submission_result.to_csv(filename, index=False)

    print(f"Файл сохранён: {filename}")
    print(submission_result.describe().to_string())

    return submission_result

## Финальная конфигурация

Финальная конфигурация собрана в одном месте, чтобы повторный запуск ноутбука создавал один и тот же `submission.csv`.

Коэффициенты применяются после ансамбля моделей и выполняют три задачи:

1. корректируют масштаб предсказаний для разных target;
2. ограничивают физически невозможные отрицательные значения;
3. стабилизируют верхний хвост распределения, особенно для `SI`.


In [ ]:
FINAL_IC50_MULT = BEST_IC50_MULT
FINAL_CC50_MULT = BEST_CC50_MULT
FINAL_SI_MULT = BEST_SI_MULT
FINAL_SI_CAP = BEST_SI_CAP

# Вес Ridge stacking в финальном blended-прогнозе.
# Значение одинаковое для всех target, чтобы не усложнять решение и сохранить
# понятную структуру ансамбля.
FINAL_STACK_WEIGHTS = {
    TARGET_IC50: 0.20,
    TARGET_CC50: 0.20,
    TARGET_SI: 0.20,
}

final_si_ratio_weight = oof_si_ratio_weight if USE_SI_RELATION_BLEND else 0.0

print("Финальная зафиксированная конфигурация:")
print("Множитель IC50:", FINAL_IC50_MULT)
print("Множитель CC50:", FINAL_CC50_MULT)
print("Множитель SI:", FINAL_SI_MULT)
print("Верхняя граница SI:", FINAL_SI_CAP)
print("Вес связи SI с CC50/IC50:", final_si_ratio_weight)
print("Веса stacking:", FINAL_STACK_WEIGHTS)


## Обоснование выбранного подхода

Основная идея решения — построить воспроизводимый ансамбль для трёх регрессионных целей с контролем качества через OOF-предсказания.

`IC50`, `CC50` и `SI` имеют разные масштабы и распределения, поэтому для каждой целевой переменной обучается отдельный набор моделей. На небольшой выборке одна модель может быть нестабильной, поэтому используются разные семейства алгоритмов: линейная модель `Ridge`, bagging-модели `RandomForest` и `ExtraTrees`, а также boosting-модели `HistGradientBoosting`, `CatBoost`, `LightGBM` и `XGBoost`.

Базовые модели объединяются через взвешенный blend: модели с меньшей OOF-ошибкой получают больший вес. Дополнительно используется Ridge stacking, который обучается комбинировать OOF-предсказания базовых моделей.

В финальной версии хвосты контролируются через `log1p`, неотрицательность, clipping в реалистичный диапазон и ограничение `SI`. Агрессивное квантильное срезание хвостов не используется, потому что оно может занизить редкие высокие значения.


## Ridge stacking

In [ ]:
from sklearn.linear_model import RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


def build_stacked_predictions_from_oof(
    all_results: dict[str, list[ModelResult]],
    y: pd.DataFrame,
    x_test_index: pd.Index,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Построить RidgeCV stacking по OOF/test-предсказаниям базовых моделей для каждой цели."""
    oof_stack = pd.DataFrame(index=y.index, columns=TARGETS, dtype=float)
    test_stack = pd.DataFrame(index=x_test_index, columns=TARGETS, dtype=float)
    rows = []

    for target in TARGETS:
        results = all_results[target]
        model_names = [item.name for item in results]
        oof_matrix = np.vstack([item.oof_pred for item in results]).T
        test_matrix = np.vstack([item.test_pred for item in results]).T

        # Простая RidgeCV-метамодель. StandardScaler нужен, потому что модели могут
        # иметь разный масштаб/дисперсию ошибок.
        meta = make_pipeline(
            StandardScaler(),
            RidgeCV(alphas=np.logspace(-4, 4, 33)),
        )
        meta.fit(oof_matrix, y[target])

        oof_pred = meta.predict(oof_matrix)
        test_pred = meta.predict(test_matrix)

        # Ограничиваем реалистичным диапазоном train, как и в базовом пайплайне.
        oof_pred = np.clip(oof_pred, y[target].min(), y[target].max())
        test_pred = np.clip(test_pred, y[target].min(), y[target].max())

        oof_stack[target] = oof_pred
        test_stack[target] = test_pred

        rows.append(
            {
                "Целевая переменная": target_display_name(target),
                "Количество базовых моделей": len(results),
                "OOF RMSE базового ансамбля": rmse(y[target], oof_predictions[target]),
                "OOF RMSE stacking": rmse(y[target], oof_pred),
                "Лучший OOF RMSE одной модели": min(item.oof_rmse for item in results),
                "Модели": ", ".join(model_names),
            }
        )

    stack_report = pd.DataFrame(rows)
    return oof_stack, test_stack, stack_report


oof_stack_predictions, test_stack_predictions, stack_report = build_stacked_predictions_from_oof(
    all_results=all_results,
    y=y,
    x_test_index=test.index,
)

print("Отчет по stacking:")
print(stack_report.to_string(index=False))
print()
print("OOF-метрика текущего взвешенного ансамбля:", competition_metric(y, oof_predictions))
print("OOF-метрика stacking:", competition_metric(y, oof_stack_predictions))


## Вывод по stacking

Ridge stacking улучшает качество по сравнению с обычным взвешенным ансамблем по всем трём целевым переменным.

На OOF-предсказаниях особенно заметное улучшение получено для `CC50`. Для `IC50` и `SI` улучшение меньше, но также положительное.

Это означает, что метамодель смогла эффективнее объединить прогнозы базовых алгоритмов, чем простое взвешенное усреднение. Поэтому stacking используется как важная часть финальной схемы.


In [ ]:
def blend_prediction_frames(
    base: pd.DataFrame,
    stack: pd.DataFrame,
    global_stack_weight: float = 0.0,
    target_stack_weights: dict[str, float] | None = None,
) -> pd.DataFrame:
    """Смешать предсказания базового ансамбля и Ridge stacking."""
    result = base[TARGETS].copy()

    if target_stack_weights is None:
        target_stack_weights = {target: global_stack_weight for target in TARGETS}

    for target in TARGETS:
        weight = float(target_stack_weights.get(target, global_stack_weight))
        result[target] = (1.0 - weight) * base[target] + weight * stack[target]

    return result


## Локальная диагностическая таблица

В таблице ниже сравниваются основные этапы решения на OOF-предсказаниях:

1. среднее по train как простой baseline;
2. взвешенный ансамбль базовых моделей;
3. ансамбль после калибровки;
4. чистый Ridge stacking;
5. финальная схема: blend + доля stacking + постобработка.

Эта таблица нужна для проверки, что усложнение пайплайна даёт локально объяснимый прирост качества.


In [ ]:
mean_baseline_oof = pd.DataFrame(index=train.index)

for target in TARGETS:
    mean_baseline_oof[target] = y[target].mean()

base_calibrated_oof = apply_final_postprocessing(
    predictions=oof_predictions,
    ic50_mult=FINAL_IC50_MULT,
    cc50_mult=FINAL_CC50_MULT,
    si_mult=FINAL_SI_MULT,
    si_cap=FINAL_SI_CAP,
    si_ratio_weight=final_si_ratio_weight,
)

stack_blended_oof_raw = blend_prediction_frames(
    base=oof_predictions,
    stack=oof_stack_predictions,
    target_stack_weights=FINAL_STACK_WEIGHTS,
)

stack_blended_oof = apply_final_postprocessing(
    predictions=stack_blended_oof_raw,
    ic50_mult=FINAL_IC50_MULT,
    cc50_mult=FINAL_CC50_MULT,
    si_mult=FINAL_SI_MULT,
    si_cap=FINAL_SI_CAP,
    si_ratio_weight=final_si_ratio_weight,
)


def metric_by_target(y_true: pd.DataFrame, y_pred: pd.DataFrame) -> dict[str, float]:
    return {
        target_display_name(target): rmse(y_true[target], y_pred[target])
        for target in TARGETS
    }


diagnostic_frames = {
    "Среднее по train (baseline)": mean_baseline_oof,
    "Взвешенный ансамбль": oof_predictions,
    "Ансамбль + калибровка": base_calibrated_oof,
    "Только Ridge stacking": oof_stack_predictions,
    "Финальная схема: ансамбль + stacking + калибровка": stack_blended_oof,
}

rows = []
for stage, frame in diagnostic_frames.items():
    row = {
        "Этап": stage,
        "Локальная OOF-метрика": competition_metric(y, frame),
    }
    row.update(metric_by_target(y, frame))
    rows.append(row)

final_diagnostic = pd.DataFrame(rows)
print(final_diagnostic.to_string(index=False))


## Вывод по сравнению этапов решения

Переход от baseline к ансамблю моделей заметно снижает OOF-ошибку: модель начинает использовать молекулярные дескрипторы, а не просто средние значения target.

Наибольший локальный прирост даёт Ridge stacking: он улучшает качество по всем трём целевым переменным, особенно для `CC50`.

Финальная схема объединяет базовый ансамбль, часть stacking-прогноза и постобработку. Она выбрана как устойчивый вариант, который сохраняет вклад разных моделей и контролирует масштаб итоговых предсказаний.


## Проверка на утечки и ограничения

В решении соблюдаются следующие ограничения:

- target-колонки удаляются из признакового пространства перед обучением;
- `test.csv` используется только для построения признаков и получения предсказаний;
- `sample_submission.csv` используется только как шаблон формата;
- внешние данные и готовые ответы не используются;
- все преобразования признаков вычисляются одинаково для train и test;
- константные признаки выбираются для удаления только по train;
- почти константные признаки анализируются, но не удаляются автоматически в финальной версии;
- итоговый файл создаётся повторным запуском ноутбука от начала до конца.


## Финальное формирование одного submission-файла

Перед сохранением удаляются старые сгенерированные файлы `submission*.csv` и `candidate_report*.csv`. После запуска должен остаться один итоговый файл:

```text
submission.csv
```


In [ ]:
# Удаляем старые сгенерированные кандидаты, чтобы после запуска остался один итоговый файл.
for pattern in ["submission*.csv", "candidate_report*.csv"]:
    for path in Path(".").glob(pattern):
        if path.name == "sample_submission.csv":
            continue
        path.unlink(missing_ok=True)

final_raw_predictions = blend_prediction_frames(
    base=test_predictions,
    stack=test_stack_predictions,
    target_stack_weights=FINAL_STACK_WEIGHTS,
)

final_predictions = apply_final_postprocessing(
    predictions=final_raw_predictions,
    ic50_mult=FINAL_IC50_MULT,
    cc50_mult=FINAL_CC50_MULT,
    si_mult=FINAL_SI_MULT,
    si_cap=FINAL_SI_CAP,
    si_ratio_weight=final_si_ratio_weight,
)

submission = save_submission(
    predictions=final_predictions,
    sample_submission=sample_submission,
    filename="submission.csv",
)

print()
print("Финальный submission создан: submission.csv")


## Финальная проверка

In [ ]:
csv_files = sorted(Path(".").glob("*.csv"))

generated_files = [
    file
    for file in csv_files
    if file.name.startswith("submission") or file.name.startswith("candidate_report")
]

print("CSV-файлы после финального сохранения:")
for file in csv_files:
    print(file)

print()
print("Сгенерированные файлы:")
for file in generated_files:
    print(file)

if [file.name for file in generated_files] != ["submission.csv"]:
    raise RuntimeError("Ожидался ровно один сгенерированный файл: submission.csv")

print()
print("Отправлять нужно этот файл:")
print("  submission.csv")


## Итоговый вывод

В ноутбуке построен воспроизводимый пайплайн для предсказания `IC50`, `CC50` и `SI`.

Решение включает:

- первичный анализ данных и распределений target;
- проверку пропусков, константных и почти константных признаков;
- обучение отдельных моделей для каждой целевой переменной;
- OOF-оценку качества;
- взвешенный blend базовых моделей;
- Ridge stacking;
- финальную постобработку предсказаний;
- сохранение единственного файла `submission.csv`.

Внешние данные не используются. `test.csv` используется только для построения признаков и получения предсказаний. `sample_submission.csv` используется только как шаблон формата ответа.
